In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import mutual_info_regression
from scipy.stats import pearsonr

data = pd.read_csv('new/processed_fpt.csv', parse_dates=['time'])
data.head()

,time,open,high,low,close,volume,Symbol,daily_return,weekly_return,monthly_return,...,out_macd,out_macd_signal,out_macd_hist,willr,tsf_10,tsf_20,trix,bbandsmiddle,bbandsupper,bbandslower
0,2018-01-01,17560.0,18330.0,17530.0,18330.0,5022160.0,FPT,0.000000,0.018549,0.051828,...,0.000000,0.000000,0.000000,25.842697,18935.777778,18383.912281,0.000000,18498.571429,19030.133368,17967.009489
1,2018-01-02,17560.0,18330.0,17530.0,18330.0,5022160.0,FPT,0.000000,0.018549,0.051828,...,0.000000,0.000000,0.000000,25.842697,18935.777778,18383.912281,0.000000,18498.571429,19030.133368,17967.009489
2,2018-01-03,18540.0,18600.0,18240.0,18330.0,2829930.0,FPT,0.000000,0.018549,0.051828,...,0.000000,0.000000,0.000000,25.842697,18935.777778,18383.912281,0.000000,18498.571429,19030.133368,17967.009489
3,2018-01-04,18390.0,18700.0,18360.0,18700.0,2784800.0,FPT,0.020185,0.018549,0.051828,...,29.515670,5.903134,23.612536,25.842697,18935.777778,18383.912281,0.003942,18498.571429,19030.133368,17967.009489
4,2018-01-05,18700.0,18730.0,18360.0,18390.0,2851450.0,FPT,-0.016578,0.018549,0.051828,...,27.574776,10.237462,17.337314,25.842697,18935.777778,18383.912281,0.007046,18498.571429,19030.133368,17967.009489


In [2]:
# Pivot dữ liệu: index = time, cột là các chỉ báo với tiền tố là tên Symbol
pivot_data = data.pivot(index='time', columns='Symbol')
pivot_data.columns = ['{}_{}'.format(sym, feat) for feat, sym in pivot_data.columns]
pivot_data.reset_index(inplace=True)
pivot_data.head()

,time,FPT_open,VN30_open,VNINDEX_open,FPT_high,VN30_high,VNINDEX_high,FPT_low,VN30_low,VNINDEX_low,...,VNINDEX_trix,FPT_bbandsmiddle,VN30_bbandsmiddle,VNINDEX_bbandsmiddle,FPT_bbandsupper,VN30_bbandsupper,VNINDEX_bbandsupper,FPT_bbandslower,VN30_bbandslower,VNINDEX_bbandslower
0,2018-01-01,17560.0,977200.0,986050.0,18330.0,992720.0,996180.0,17530.0,975490.0,984240.0,...,-0.013286,18498.571429,190667.619048,1.304739e+06,19030.133368,5.582617e+05,1.450231e+06,17967.009489,-176926.500583,1.159247e+06
1,2018-01-02,17560.0,977200.0,986050.0,18330.0,992720.0,996180.0,17530.0,975490.0,984240.0,...,-0.089761,18498.571429,230916.190476,1.290412e+06,19030.133368,7.374797e+05,1.488869e+06,17967.009489,-275647.303047,1.091955e+06
2,2018-01-03,18540.0,996230.0,999860.0,18600.0,1011840.0,1010210.0,18240.0,992720.0,995770.0,...,-0.183449,18498.571429,271728.571429,1.276475e+06,19030.133368,8.783243e+05,1.510510e+06,17967.009489,-334867.191961,1.042440e+06
3,2018-01-04,18390.0,1007600.0,1009370.0,18700.0,1014170.0,1019750.0,18360.0,1004660.0,1005670.0,...,-0.282540,18498.571429,313111.904762,1.263643e+06,19030.133368,9.970684e+05,1.522930e+06,17967.009489,-370844.628576,1.004356e+06
4,2018-01-05,18700.0,1014420.0,1020340.0,18730.0,1014420.0,1020600.0,18360.0,1005100.0,1010650.0,...,-0.381965,18498.571429,354170.476190,1.250072e+06,19030.133368,1.096866e+06,1.530829e+06,17967.009489,-388524.792820,9.693148e+05


In [ ]:
data.columns

In [3]:
full_features = ['open', 'high', 'low', 'close', 'volume',
    'daily_return', 'weekly_return', 'monthly_return', 'daily_volatility',
    'weekly_volatility', 'monthly_volatility', 'daily_liquidity',
    'weekly_liquidity', 'monthly_liquidity', 'high_minus_close',
    'low_minus_open', 'cumulativ_return', 'wma_3', 'wma_7', 'wma_14',
    'wma_21', 'wma_50', 'wma_100', 'obv', 'rsi_6', 'rsi_12', 'rsi_14',
    'stoch_rsi_6', 'stoch_rsi_12', 'stoch_rsi_14', 'sma_3', 'sma_7',
    'sma_14', 'sma_21', 'sma_50', 'sma_100', 'ema_6', 'ema_12', 'atr_14',
    'mfi_14', 'adx_14', 'adx_20', 'mom_1', 'mom_3', 'cci_12', 'cci_20',
    'rocr_3', 'rocr_12', 'out_macd', 'out_macd_signal', 'out_macd_hist',
    'willr', 'tsf_10', 'tsf_20', 'trix', 'bbandsmiddle', 'bbandsupper',
    'bbandslower'
]

def get_features(df, symbol):
    cols = [f"{symbol}_{feat}" for feat in full_features]
    return df[cols]

In [4]:
# Xây dựng ma trận đặc trưng theo thứ tự: FPT, VN30, VNINDEX
X_fpt = get_features(pivot_data, 'FPT')
X_vn30 = get_features(pivot_data, 'VN30')
X_vnindex = get_features(pivot_data, 'VNINDEX')
X = pd.concat([X_fpt, X_vn30, X_vnindex], axis=1)
print("Kích thước ma trận X:", X.shape)

Kích thước ma trận X: (2588, 174)


1 day prediction

In [ ]:
# prices = pivot_data['FPT_close']
# Y = np.where(prices.shift(-1) > prices, 1, -1)
# X = X.iloc[:-1, :]
# Y = Y[:-1]

3 days prediction

In [ ]:
prices = pivot_data['FPT_close']
sma = prices.rolling(window=30).mean()

Y = np.where(sma.shift(-30) > sma, 1, -1)

valid_index = (~sma.isna()) & (~sma.shift(-20).isna())
X = X.loc[valid_index].reset_index(drop=True)
Y = Y[valid_index]

print("Kích thước ma trận X sau khi loại bỏ NaN:", X.shape)
print("Kích thước vector Y:", Y.shape)

In [ ]:
print("Kích thước vector Y:", Y.shape)

In [ ]:
window_train = 1000
window_test = 30
step = 5
n = len(X)
n_splits = (n - window_train - window_test) // step + 1
print("Số cửa sổ rolling:", n_splits)

Feature Selection via Correction and p-value

In [ ]:
def select_features_by_pvalue(X_scaled, Y, p_threshold=0.05, corr_threshold=0.1):
    selected_indices = []
    n_features = X_scaled.shape[1]
    for j in range(n_features):
        r, p = pearsonr(X_scaled[:, j], Y)
        if p < p_threshold and abs(r) > corr_threshold:
            selected_indices.append(j)
    return selected_indices

scaler_tmp = StandardScaler()
X_scaled_tmp = scaler_tmp.fit_transform(X)
sel_idx = select_features_by_pvalue(X_scaled_tmp, Y)
print("Số feature được chọn toàn bộ:", len(sel_idx))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

n_features = X_scaled.shape[1]
corrs = []
p_values = []
feature_names = X.columns.tolist()

for j in range(n_features):
    r, p = pearsonr(X_scaled[:, j], Y)
    corrs.append(r)
    p_values.append(p)

corrs = np.array(corrs)
p_values = np.array(p_values)

# Vẽ scatter plot: x = hệ số tương quan, y = p-value (log scale)
plt.figure(figsize=(12, 6))
plt.scatter(corrs, p_values, c='blue', label='Feature points')
plt.axhline(0.05, color='red', linestyle='--', label='p-value = 0.05')
plt.axvline(0.1, color='green', linestyle='--', label='corr = ±0.1')
plt.axvline(-0.1, color='green', linestyle='--')

plt.xlabel("Pearson Correlation")
plt.ylabel("p-value (log scale)")
plt.yscale("log")
plt.title("Scatter Plot: Pearson Correlation vs p-value for each feature")
plt.legend()

for i, name in enumerate(feature_names):
    plt.annotate(name, (corrs[i], p_values[i]), fontsize=8, alpha=0.7)

plt.show()

In [ ]:
accuracies = []
best_params_list = []


param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1]
}

for i in range(n_splits):
    start = i * step
    end_train = start + window_train
    end_test = end_train + window_test
    
    X_train = X.iloc[start:end_train, :].values
    Y_train = Y[start:end_train]
    X_test = X.iloc[end_train:end_test, :].values
    Y_test = Y[end_train:end_test]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Chọn lọc feature dựa trên p-value và correlation
    selected_indices = select_features_by_pvalue(X_train_scaled, Y_train, p_threshold=0.05, corr_threshold=0.1)
    if len(selected_indices) == 0:
        selected_indices = list(range(X_train_scaled.shape[1]))
    
    X_train_fs = X_train_scaled[:, selected_indices]
    X_test_fs = X_test_scaled[:, selected_indices]
    
    svc = SVC(kernel='rbf')
    cv = StratifiedKFold(n_splits=5)
    grid = GridSearchCV(svc, param_grid, cv=cv, scoring='accuracy')
    grid.fit(X_train_fs, Y_train)
    best_model = grid.best_estimator_
    
    Y_pred = best_model.predict(X_test_fs)
    acc = accuracy_score(Y_test, Y_pred)
    accuracies.append(acc)
    best_params_list.append(grid.best_params_)
    
    print(f"Cửa sổ {i+1}: Độ chính xác = {acc:.4f}, Tham số tốt nhất = {grid.best_params_}, Số feature được chọn = {len(selected_indices)}")

Feature Selection via Randomised Forest

In [ ]:
# accuracies = []
# best_params_list = []

# param_grid = {
#     'C': [0.1, 1, 10, 100],
#     'gamma': [0.001, 0.01, 0.1, 1]
# }

# for i in range(n_splits):
#     start = i * step
#     end_train = start + window_train
#     end_test = end_train + window_test
#     X_train = X.iloc[start:end_train, :].values
#     Y_train = Y[start:end_train]
#     X_test = X.iloc[end_train:end_test, :].values
#     Y_test = Y[end_train:end_test]
    
#     scaler = StandardScaler()
#     X_train_scaled = scaler.fit_transform(X_train)
#     X_test_scaled = scaler.transform(X_test)
    
#     # ---------------- Feature Selection ----------------
#     etc = ExtraTreesClassifier(n_estimators=100, random_state=42)
#     etc.fit(X_train_scaled, Y_train)
    
#     importances = etc.feature_importances_
#     indices = np.argsort(importances)[::-1]
    
#     k = int(np.ceil(0.5 * X_train_scaled.shape[1]))
#     selected_indices = indices[:k]
    
#     X_train_fs = X_train_scaled[:, selected_indices]
#     X_test_fs = X_test_scaled[:, selected_indices]
    
#     svc = SVC(kernel='rbf')
#     cv = StratifiedKFold(n_splits=5)
#     grid = GridSearchCV(svc, param_grid, cv=cv, scoring='accuracy')
#     grid.fit(X_train_fs, Y_train)
#     best_model = grid.best_estimator_
    
#     Y_pred = best_model.predict(X_test_fs)
#     acc = accuracy_score(Y_test, Y_pred)
#     accuracies.append(acc)
#     best_params_list.append(grid.best_params_)
    
#     print(f"Cửa sổ {i+1}: Độ chính xác = {acc:.4f}, Tham số tốt nhất = {grid.best_params_}, Số feature đã chọn = {k}")

# Testing

In [ ]:
avg_accuracy = np.mean(accuracies)
print("Độ chính xác trung bình trên tập test:", avg_accuracy)

# Vẽ đồ thị độ chính xác theo từng cửa sổ rolling
plt.figure(figsize=(10,5))
plt.plot(range(1, n_splits+1), accuracies, marker='o')
plt.title("Độ chính xác trên từng cửa sổ rolling (có feature selection)")
plt.xlabel("Số cửa sổ")
plt.ylabel("Accuracy")
plt.grid(True)
plt.show()